# Driving complaints, collisions, and stops by police beat

## Preparing the 2024–2025 comparison

Count distinct vehicle stops and collision events using a common local-date window and a fixed City of Berkeley beat map. Complaint counts and rank correlations await the shared calls data.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib.path import Path as PolygonPath

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
data = root / "data"
features = json.loads((data / "boundaries/police_beats.geojson").read_text())["features"]
stops = pd.read_csv(data / "processed/vehicle_stops.csv")
collisions = pd.read_csv(data / "processed/collisions.csv")

In [ ]:
def assign_beats(points, features):
    """Return a Series of beat IDs or exclusion reasons for an N-by-2 lon/lat array."""
    valid = np.isfinite(points).all(axis=1)
    assignments = np.full(len(points), "outside", dtype=object)
    assignments[~valid] = "missing"
    for feature in features:
        beat = str(feature["properties"]["Sector"])
        geometry = feature["geometry"]
        parts = geometry["coordinates"]
        if geometry["type"] == "Polygon":
            parts = [parts]
        elif geometry["type"] != "MultiPolygon":
            raise ValueError("Expected polygon geometry")
        inside = np.zeros(len(points), dtype=bool)
        boundary = np.zeros(len(points), dtype=bool)
        for shell, *holes in parts:
            part = np.ones(len(points), dtype=bool)
            for index, ring in enumerate([shell, *holes]):
                path = PolygonPath(ring)
                # Opposite tiny radii flag exact edge points regardless of winding.
                a = path.contains_points(points, radius=1e-10)
                b = path.contains_points(points, radius=-1e-10)
                boundary |= a ^ b
                selected = a & b
                part &= selected if index == 0 else ~selected
            inside |= part
        ambiguous = boundary | (inside & (assignments != "outside"))
        assignments[inside & valid] = beat
        assignments[ambiguous & valid] = "ambiguous"
    return pd.Series(assignments)


def in_window(frame, column, fmt):
    """Select DataFrame rows whose local date string falls in 2024 or 2025."""
    dates = pd.to_datetime(frame[column], format=fmt)
    return frame.loc[dates.dt.year.isin([2024, 2025])]


stops = in_window(stops, "StopDate", "%m/%d/%Y")
collisions = in_window(collisions, "Date", "%Y-%m-%d")
stop_points = stops[["Longitude", "Latitude"]].to_numpy()
# Convert spherical Web Mercator metres to longitude and latitude in degrees.
radius_m = 6378137
lon = np.degrees(collisions["x"] / radius_m)
lat = np.degrees(np.arctan(np.sinh(collisions["y"] / radius_m)))
counts = pd.DataFrame({
    "vehicle_stops": assign_beats(stop_points, features).value_counts(),
    "collisions": assign_beats(np.column_stack([lon, lat]), features).value_counts(),
}).fillna(0).astype(int)
display(counts.reindex(["missing", "outside", "ambiguous"], fill_value=0))

In [ ]:
beats = sorted({str(f["properties"]["Sector"]) for f in features}, key=int)
table = counts.reindex(beats, fill_value=0).rename_axis("beat")
excluded = counts.loc[~counts.index.isin(beats)].sum()
assert (table.sum() + excluded).tolist() == [len(stops), len(collisions)]
table.to_csv(data / "processed/a5_driving_by_beat.csv")
display(table)